In [1]:
import warnings
warnings.simplefilter(action='ignore')

In [2]:
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import scarches as sca
import sys

 captum (see https://github.com/pytorch/captum).


In [3]:
sc.set_figure_params(frameon=False)
sc.set_figure_params(dpi=200)
sc.set_figure_params(figsize=(4, 4))
torch.set_printoptions(precision=3, sci_mode=False, edgeitems=7)

In [4]:
perc = 1
top_genes = None
condition = 'cancer_subtype'

adata_path = f'/home/lgolinelli/git/scarches-1/notebooks/GRN-VAE/anndata_inputs/adata_top_{perc}_top_genes_{top_genes}.h5ad'
model_path = f'/home/lgolinelli/git/scarches-1/notebooks/GRN-VAE/trained_models/model_{perc}_top_genes_{top_genes}_{condition}'

In [5]:
adata = sc.read_h5ad(adata_path)

In [6]:
adata.obs["placeholder_condition"] = "placeholder"

In [7]:
vae = sca.models.EXPIMAP.load(model_path, adata, map_location='cpu')

AnnData object with n_obs × n_vars = 4544 × 15984
    obs: 'cancer_subtype', 'placeholder_condition'
    var: 'gene_ids', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'gene_idx', 'counts_in_terms', 'gene_as_regulator'
    uns: 'cancer_subtype_colors', 'hvg', 'log1p', 'neighbors', 'nsd2_as_regulator', 'nsd2_as_target', 'pca', 'terms', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'I', 'PCs'
    layers: 'X_normalized', 'counts'
    obsp: 'connectivities', 'distances'

INITIALIZING NEW NETWORK..............
Encoder Architecture:
	Input Layer in, out and cond: 15984 300 3
	Hidden Layer 1 in/out: 300 300
	Hidden Layer 2 in/out: 300 300
	Mean/Var Layer in/out: 300 2532
Decoder Architecture:
	Masked linear layer in, ext_m, ext, cond, out:  2532 0 0 3 15984
	with hard mask.
Last Decoder layer: softmax


In [8]:
full_model = vae.model
encoder = full_model.encoder
decoder = full_model.decoder

In [9]:
conditions = adata.obs["placeholder_condition"].values
conditions

array(['placeholder', 'placeholder', 'placeholder', 'placeholder',
       'placeholder', 'placeholder', 'placeholder', ..., 'placeholder',
       'placeholder', 'placeholder', 'placeholder', 'placeholder',
       'placeholder', 'placeholder'], dtype=object)

In [10]:
'''adata.obsm['z1_mean'] = enc_out['z1_mean'].detach().numpy()
sc.pp.neighbors(adata, use_rep='z1_mean')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['cancer_subtype'])'''

"adata.obsm['z1_mean'] = enc_out['z1_mean'].detach().numpy()\nsc.pp.neighbors(adata, use_rep='z1_mean')\nsc.tl.umap(adata)\nsc.pl.umap(adata, color=['cancer_subtype'])"

In [11]:
'''sc.pp.neighbors(adata, use_rep='X')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['cancer_subtype'])'''

"sc.pp.neighbors(adata, use_rep='X')\nsc.tl.umap(adata)\nsc.pl.umap(adata, color=['cancer_subtype'])"

In [12]:
'''adata.obsm['sample'] = dec_out['sample'].cpu().numpy()
sc.pp.neighbors(adata, use_rep='sample')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['cancer_subtype'])'''

"adata.obsm['sample'] = dec_out['sample'].cpu().numpy()\nsc.pp.neighbors(adata, use_rep='sample')\nsc.tl.umap(adata)\nsc.pl.umap(adata, color=['cancer_subtype'])"

In [13]:
'''adata.obsm['dec_mean'] = dec_out['dec_mean'].detach().numpy()
sc.pp.neighbors(adata, use_rep='dec_mean')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['cancer_subtype'])'''

"adata.obsm['dec_mean'] = dec_out['dec_mean'].detach().numpy()\nsc.pp.neighbors(adata, use_rep='dec_mean')\nsc.tl.umap(adata)\nsc.pl.umap(adata, color=['cancer_subtype'])"

In [14]:
pert_gene_adata = vae.perturb_genes(
    adata=adata,
    genes='Nsd2',
    perturb_type='KO',
    group_key='cancer_subtype',
    category='CRPC-NE',
    obs_key='perturbation',
    condition_key='cancer_subtype',
    new_condition=None
)

In [15]:
vae.e_distance(pert_gene_adata, cluster_key='cancer_subtype_perturb', embedding_key='z1_mean_all', n_jobs=-1)


                 CRPC-NE  CRPC-NE_perturb  CRPC-adeno    DNPC
CRPC-NE           0.0000           0.0000      2.9087  6.8117
CRPC-NE_perturb                    0.0000      2.9081  6.8131
CRPC-adeno                                     0.0000 11.0093
DNPC                                                   0.0000


,CRPC-NE,CRPC-NE_perturb,CRPC-adeno,DNPC
CRPC-NE,0.0,0.000036,2.908699,6.811724
CRPC-NE_perturb,NaN,0.000000,2.908144,6.813114
CRPC-adeno,NaN,NaN,0.000000,11.009291
DNPC,NaN,NaN,NaN,0.000000


In [16]:
pert_gene_adata.obsm['perturbed_dec_mean'] = pert_gene_adata.layers['perturbed_dec_mean']
vae.e_distance(pert_gene_adata, cluster_key='cancer_subtype_perturb', embedding_key='perturbed_dec_mean', n_jobs=-1)


                 CRPC-NE  CRPC-NE_perturb  CRPC-adeno   DNPC
CRPC-NE           0.0000           0.0000      0.0240 0.0131
CRPC-NE_perturb                    0.0000      0.0240 0.0131
CRPC-adeno                                     0.0000 0.0132
DNPC                                                  0.0000


,CRPC-NE,CRPC-NE_perturb,CRPC-adeno,DNPC
CRPC-NE,0.0,2.048910e-08,0.024033,0.013138
CRPC-NE_perturb,NaN,0.000000e+00,0.024027,0.013133
CRPC-adeno,NaN,NaN,0.000000,0.013197
DNPC,NaN,NaN,NaN,0.000000


In [17]:
pert_gene_adata = vae.perturb_genes(
    adata=adata,
    genes='Nsd2',
    perturb_type='Overexpression',
    group_key='cancer_subtype',
    category='CRPC-NE',
    obs_key='perturbation',
    condition_key='cancer_subtype',
    new_condition=None
)


In [18]:
pert_gene_adata.obsm['perturbed_dec_mean'] = pert_gene_adata.layers['perturbed_dec_mean']
vae.e_distance(pert_gene_adata, cluster_key='cancer_subtype_perturb', embedding_key='z1_mean_all', n_jobs=-1)

                 CRPC-NE  CRPC-NE_perturb  CRPC-adeno    DNPC
CRPC-NE           0.0000           0.0002      2.9087  6.8117
CRPC-NE_perturb                    0.0000      2.9109  6.8081
CRPC-adeno                                     0.0000 11.0093
DNPC                                                   0.0000


,CRPC-NE,CRPC-NE_perturb,CRPC-adeno,DNPC
CRPC-NE,0.0,0.000153,2.908699,6.811724
CRPC-NE_perturb,NaN,0.000000,2.910923,6.808092
CRPC-adeno,NaN,NaN,0.000000,11.009291
DNPC,NaN,NaN,NaN,0.000000


In [19]:
vae.e_distance(pert_gene_adata, cluster_key='cancer_subtype_perturb', embedding_key='z1_mean_all', n_jobs=-1)


                 CRPC-NE  CRPC-NE_perturb  CRPC-adeno    DNPC
CRPC-NE           0.0000           0.0002      2.9087  6.8117
CRPC-NE_perturb                    0.0000      2.9109  6.8081
CRPC-adeno                                     0.0000 11.0093
DNPC                                                   0.0000


,CRPC-NE,CRPC-NE_perturb,CRPC-adeno,DNPC
CRPC-NE,0.0,0.000153,2.908699,6.811724
CRPC-NE_perturb,NaN,0.000000,2.910923,6.808092
CRPC-adeno,NaN,NaN,0.000000,11.009291
DNPC,NaN,NaN,NaN,0.000000


In [20]:
pert_gp_adata = vae.perturb_gps(
    adata=adata,
    programs='Nsd2',
    perturb_type='KO',
    group_key='cancer_subtype',
    category='CRPC-NE',
    obs_key='perturbation',
    condition_key='cancer_subtype',
    new_condition=None
)

pert_gp_adata.obsm['perturbed_dec_mean'] = pert_gp_adata.layers['perturbed_dec_mean']
vae.e_distance(pert_gp_adata, cluster_key='cancer_subtype_perturb', embedding_key='perturbed_dec_mean', n_jobs=-1)


                 CRPC-NE  CRPC-NE_perturb  CRPC-adeno   DNPC
CRPC-NE           0.0000           0.0000      0.0240 0.0131
CRPC-NE_perturb                    0.0000      0.0240 0.0131
CRPC-adeno                                     0.0000 0.0132
DNPC                                                  0.0000


,CRPC-NE,CRPC-NE_perturb,CRPC-adeno,DNPC
CRPC-NE,0.0,0.0,0.024033,0.013138
CRPC-NE_perturb,NaN,0.0,0.024033,0.013138
CRPC-adeno,NaN,NaN,0.000000,0.013197
DNPC,NaN,NaN,NaN,0.000000


In [23]:
pert_gp_adata

AnnData object with n_obs × n_vars = 5709 × 15984
    obs: 'cancer_subtype', 'placeholder_condition', 'cancer_subtype_perturb', 'perturbation'
    uns: 'df_pert_genes_diffs', 'df_pert_programs_diffs'
    obsm: 'X_pca', 'X_umap', 'X_expimap_raw', 'X_expimap_pert', 'perturbed_dec_mean'
    layers: 'X_normalized', 'counts', 'perturbed_dec_mean'

In [24]:
vae.e_distance(pert_gp_adata, cluster_key='cancer_subtype_perturb', embedding_key='X_expimap_pert', n_jobs=-1)


                 CRPC-NE  CRPC-NE_perturb  CRPC-adeno    DNPC
CRPC-NE           0.0000           0.0000      2.9087  6.8117
CRPC-NE_perturb                    0.0000      2.9087  6.8117
CRPC-adeno                                     0.0000 11.0093
DNPC                                                   0.0000


,CRPC-NE,CRPC-NE_perturb,CRPC-adeno,DNPC
CRPC-NE,0.0,0.000002,2.908699,6.811724
CRPC-NE_perturb,NaN,0.000000,2.908697,6.811722
CRPC-adeno,NaN,NaN,0.000000,11.009291
DNPC,NaN,NaN,NaN,0.000000


In [21]:
pert_gene_adata.uns['df_pert_genes_diffs']

,gene_index,gene_name,raw_diff,abs_diff
0,15966,mt-Atp6,-2.003461e-05,2.003461e-05
1,6107,Gapdh,-1.570256e-05,1.570256e-05
2,8198,Rpl41,1.144595e-05,1.144595e-05
3,7422,Rplp2,-7.329043e-06,7.329043e-06
4,10185,Eef1a1,-6.248709e-06,6.248709e-06
...,...,...,...,...
15979,3914,Ccdc17,1.421085e-12,1.421085e-12
15980,10963,Tmem220,1.250555e-12,1.250555e-12
15981,2067,Ripor3,-1.136868e-12,1.136868e-12
15982,936,Cd46,7.958079e-13,7.958079e-13


In [22]:
pert_gene_adata.uns['df_pert_programs_diffs']

,program_index,program_name,raw_diff,abs_diff
0,734,Hist1h2ap,9.520747e-03,9.520747e-03
1,645,Grhl2,-6.877400e-03,6.877400e-03
2,1959,Tipin,6.750130e-03,6.750130e-03
3,1979,Top2a,6.109357e-03,6.109357e-03
4,378,Dek,-5.932935e-03,5.932935e-03
...,...,...,...,...
2527,1088,Med31,-1.755252e-07,1.755252e-07
2528,210,Casp8ap2,-1.687476e-07,1.687476e-07
2529,502,Exo5,1.672888e-07,1.672888e-07
2530,821,Hoxa7,-9.941868e-08,9.941868e-08
